In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import ast
import numpy as np
import sinter
import datetime

In [ ]:
def load_data(file_pattern='../out/collection/bias_comparison*.csv'):
    import glob
    file_list = glob.glob(file_pattern)
    if not file_list:
        raise FileNotFoundError(f'No files matched: {file_pattern}')
    df = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)

    # Group by strong_id to merge repeated runs
    df = df.groupby('strong_id').agg({
        'shots': 'sum',
        'errors': 'sum',
        'discards': 'sum',
        'seconds': 'sum',
        'decoder': 'first',
        'json_metadata': 'first',
    }).reset_index()

    # Parse metadata
    df['metadata'] = df['json_metadata'].apply(ast.literal_eval)
    metadata_df = pd.json_normalize(df['metadata'])
    df = pd.concat([df, metadata_df], axis=1)
    return df

df = load_data()
df[['n', 'expected_distance', 'bell_bias', 'p', 'shots', 'errors']].sort_values(['bell_bias', 'n', 'p'])

In [ ]:
# Build sinter TaskStats for plotting
sinter_stats = []
for _, row in df.iterrows():
    sinter_stats.append(sinter.TaskStats(
        strong_id=row['strong_id'],
        decoder=row['decoder'],
        json_metadata=ast.literal_eval(row['json_metadata']),
        shots=int(row['shots']),
        errors=int(row['errors']),
        discards=int(row['discards']),
        seconds=float(row['seconds']),
    ))

# Colors for each (n, bias) group
COLORS = {
    (64, 1.0): '#76b7b2',
    (256, 1.0): '#4e79a7',
    (576, 1.0): '#3B2B81',
    (64, 10.0): '#edc948',
    (256, 10.0): '#f28e2b',
    (576, 10.0): '#e15759',
}

def group_func(stat):
    n = stat.json_metadata['n']
    d = stat.json_metadata['expected_distance']
    bias = stat.json_metadata.get('bell_bias', 1)
    return f'[[{n}, 10, {d}]] bias={int(bias)}'

def plot_args_func(index, group_key, group_stats):
    # Extract n and bias from group_key
    stat = group_stats[0]
    n = stat.json_metadata['n']
    bias = stat.json_metadata.get('bell_bias', 1)
    color = COLORS.get((n, float(bias)), '#999999')
    linestyle = '-' if bias == 1 else '--'
    return {'color': color, 'linestyle': linestyle}

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

sinter.plot_error_rate(
    ax=ax,
    stats=sinter_stats,
    x_func=lambda stat: stat.json_metadata['p'],
    group_func=group_func,
    plot_args_func=plot_args_func,
    failure_values_func=lambda stat: stat.json_metadata['k'],
)

ax.loglog()
ax.set_xlabel('Physical Error Rate (p)')
ax.set_ylabel('Logical Error Rate per round')
ax.set_title('Bell bias comparison: bias=1 (solid) vs bias=10 (dashed)')
ax.grid(which='major', linestyle='-', linewidth=0.5, alpha=0.7)
ax.grid(which='minor', linestyle=':', linewidth=0.3, alpha=0.5)

# Sort legend by n then bias
handles, labels = ax.get_legend_handles_labels()
import re
def sort_key(label):
    n_match = re.search(r'\[\[(\d+)', label)
    bias_match = re.search(r'bias=(\d+)', label)
    return (int(n_match.group(1)), int(bias_match.group(1)))
sorted_pairs = sorted(zip(handles, labels), key=lambda x: sort_key(x[1]))
sorted_handles, sorted_labels = zip(*sorted_pairs)
ax.legend(sorted_handles, sorted_labels, loc='lower right', frameon=True, edgecolor='black', fancybox=False)

fig.tight_layout()
fig.set_dpi(150)

out_dir = '../out/figures'
import pathlib
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
date = datetime.datetime.now().strftime('%Y%m%d')
fig.savefig(f'{out_dir}/{date}_bias_comparison.pdf', bbox_inches='tight')
plt.show()